# PhysAR-IDS -- NB-SHAP (v6): which features does the tuned XGBoost actually use, per class?

**Why.** Two reviewer points depend on this notebook.
* **Issue 3 (physics immunity may be built into the setup).** The attacker may only change the 60 cyber features. If FDI, ransomware and reverseshell are recognised mainly from the 33 physical features, a 0% evasion of those classes under a small budget follows from the threat model, not from physics. The reviewer asked for SHAP results showing which features drive each class.
* **Issue 6 (explainability claimed, never shown).** The dissertation lists explainability as a contribution and a reason for choosing XGBoost; no result backs it.

**What it does** (tuned XGBoost from NB2, the v6 temporal split, the same 5 seeds):
1. Exact TreeSHAP values from XGBoost itself (`pred_contribs=True`; no extra package), on a fixed class-balanced test sample. For each true class, attribution = contribution to that class's own output (log-odds), mean |value| per feature.
2. Share of total attribution carried by each feature group: raw physical (11), derived physical (4), temporal physical (18), cyber 1 s (10), cyber 2 s (25), cyber 5 s (25); and physical vs cyber overall.
3. Top-10 features per class with their mean shift from benign (in standard deviations), so a suspicious electrical signature of an attack class (a possible dataset artefact) is visible rather than assumed.
4. Stability: how much the importance ranking changes between seeds.
5. A necessity check that attribution alone cannot give: tuned models trained on physical-only, cyber-only and all features, with per-class F1.

**Limits to keep in mind.** SHAP is attribution, not causation; correlated features share credit, so a group's share is not the loss you would see if you removed it (check 5 covers that). Values are in the model's log-odds space.

## 0. Setup

In [ ]:
import os, sys, random, time, json, platform, warnings
import numpy as np, pandas as pd, joblib
import xgboost as xgb
from scipy import stats
from sklearn.metrics import f1_score
import matplotlib; import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

BASE_DIR    = r'C:\Users\user\Desktop\smartrid1'
PROJECT_DIR = os.path.join(BASE_DIR, 'SmartGrid_Dissertation')
os.makedirs(PROJECT_DIR, exist_ok=True); os.chdir(PROJECT_DIR)

PIPELINE_VERSION = 'v6'
SEEDS = [42, 7, 123, 2024, 99]
SMOKE_TEST = False
N_PER_CLASS = 1500          # rows per true class for the SHAP sample (fixed across seeds)
SHAP_CHUNK = 2048

with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    _m = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2); _m.fit(np.zeros((10, 2)), np.array([0, 1] * 5))
    _cpu = any('No visible GPU' in str(w.message) for w in _caught)
BACKEND = {'tree_method': 'hist'} if _cpu else {'tree_method': 'hist', 'device': 'cuda'}
BACKEND_NAME = 'cpu(hist)' if _cpu else 'gpu(cuda,hist)'
print(f'xgboost {xgb.__version__} backend: {BACKEND_NAME}')
print('Working directory:', os.getcwd())

## 1. Load v6 data and the NB2 tuned configuration

In [ ]:
data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')
assert data.get('PIPELINE_VERSION') == PIPELINE_VERSION and data.get('TEST_FRAC') == 0.15, 'version mismatch'
X_train = data['X_train_scaled'].astype(np.float32); X_test = data['X_test_scaled'].astype(np.float32)
y_train = np.asarray(data['y_train']); y_test = np.asarray(data['y_test'])
FEATURE_COLS = list(data['FEATURE_COLS']); CYBER_IDX = np.asarray(data['CYBER_IDX']); N_FEAT = X_train.shape[1]
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']
best_params = json.load(open(f'xgb_tuned_best_params_{PIPELINE_VERSION}.json'))
COMMON = dict(objective='multi:softprob', num_class=6, eval_metric='mlogloss', n_jobs=-1, **BACKEND)

RAW = ['Freq', 'Theta', 'V_A', 'V_B', 'V_C', 'I_A', 'I_B', 'I_C', 'ActivePower', 'ReactivePower', 'BreakerStatus']
DERIVED = ['voltage_imbalance', 'current_imbalance', 'power_factor', 'freq_deviation']
def group_of(c):
    if c in RAW: return 'raw physical'
    if c in DERIVED: return 'derived physical'
    if c.startswith(('d_', 'roll_')): return 'temporal physical'
    if c.startswith('cyber_x2_'): return 'cyber 2 s'
    if c.startswith('cyber_x5_'): return 'cyber 5 s'
    if c.startswith('cyber_'): return 'cyber 1 s'
    raise ValueError(c)
FEAT_GROUP = np.array([group_of(c) for c in FEATURE_COLS])
GROUP_ORDER = ['raw physical', 'derived physical', 'temporal physical', 'cyber 1 s', 'cyber 2 s', 'cyber 5 s']
IS_CYBER = np.zeros(N_FEAT, bool); IS_CYBER[CYBER_IDX] = True
assert all((FEAT_GROUP == g).sum() > 0 for g in GROUP_ORDER) and IS_CYBER.sum() == 60 and ((FEAT_GROUP == 'cyber 1 s') | (FEAT_GROUP == 'cyber 2 s') | (FEAT_GROUP == 'cyber 5 s')).sum() == 60
print({g: int((FEAT_GROUP == g).sum()) for g in GROUP_ORDER}, '| physical', int((~IS_CYBER).sum()), '| cyber', int(IS_CYBER.sum()))

if SMOKE_TEST:
    N_PER_CLASS = 120
rng = np.random.RandomState(0)
sel = np.concatenate([rng.choice(np.where(y_test == c)[0], size=min(N_PER_CLASS, int((y_test == c).sum())), replace=False) for c in range(6)])
X_s, y_s = X_test[sel], y_test[sel]
print(f'SHAP sample: {len(sel)} test rows, per class {np.bincount(y_s).tolist()}')
print('Test class counts:', np.bincount(y_test).tolist())
# mean shift of each feature from benign (test set, standard-deviation units) -- artefact check
benign_mean = X_test[y_test == 0].mean(0)
SHIFT = np.stack([X_test[y_test == c].mean(0) - benign_mean for c in range(6)])      # (6, F)
const_mask = X_train.std(0) < 1e-9; print('Constant features in train:', [FEATURE_COLS[i] for i in np.where(const_mask)[0]])

## 2. Exact TreeSHAP from XGBoost, with an additivity check

In [ ]:
def tree_shap(clf, X):
    """(n, 6, F+1) contributions in log-odds; last column = bias. Uses the fitted backend, falls back to CPU."""
    bst = clf.get_booster(); out = []
    for i in range(0, len(X), SHAP_CHUNK):
        dm = xgb.DMatrix(np.ascontiguousarray(X[i:i + SHAP_CHUNK]))
        try: out.append(bst.predict(dm, pred_contribs=True))
        except Exception as e:
            bst.set_param({'device': 'cpu'}); out.append(bst.predict(dm, pred_contribs=True))
    return np.concatenate(out)

def check_additivity(clf, X, phi, tol=1e-3):
    margin = clf.get_booster().predict(xgb.DMatrix(np.ascontiguousarray(X)), output_margin=True)
    err = np.abs(phi.sum(2) - margin).max(); assert err < tol, f'SHAP additivity violated: {err}'
    return float(err)
print('helpers defined')

## 3. Per-seed run: fit tuned model, SHAP per class, feature-group models

In [ ]:
PHYS_IDX = np.where(~IS_CYBER)[0]
SUBSETS = {'all features': np.arange(N_FEAT), 'physical only (33)': PHYS_IDX, 'cyber only (60)': CYBER_IDX}
RES = {}; t_all = time.time()
for seed in SEEDS:
    pf = f'shap_seed{seed}_{PIPELINE_VERSION}.pkl'
    cfg_fp = json.dumps([best_params, N_PER_CLASS, seed, len(sel), float(X_s[:5].sum()), SMOKE_TEST], sort_keys=True, default=str)
    if os.path.exists(pf):
        saved = joblib.load(pf)
        if saved.get('fp') == cfg_fp: RES[seed] = saved['res']; print(f'seed {seed}: result file found -- skipped'); continue
    t0 = time.time(); print(f'=== seed {seed}', flush=True)
    clf = xgb.XGBClassifier(**best_params, **COMMON, random_state=seed); clf.fit(X_train, y_train, verbose=False)
    f1_all = f1_score(y_test, clf.predict(X_test), average='weighted')
    phi = tree_shap(clf, X_s); err = check_additivity(clf, X_s, phi)
    # own-class attribution: for rows of true class c, |phi| on the class-c output, averaged over rows
    imp = np.stack([np.abs(phi[y_s == c, c, :N_FEAT]).mean(0) for c in range(6)])            # (6, F)
    signed = np.stack([phi[y_s == c, c, :N_FEAT].mean(0) for c in range(6)])                 # (6, F)
    glob_imp = np.abs(phi[:, :, :N_FEAT]).mean((0, 1))
    # necessity check: tuned model on a feature subset, per-class F1
    sub = {}
    for name, idx in SUBSETS.items():
        if name == 'all features': pred = clf.predict(X_test)
        else:
            m = xgb.XGBClassifier(**best_params, **COMMON, random_state=seed); m.fit(X_train[:, idx], y_train, verbose=False); pred = m.predict(X_test[:, idx])
        sub[name] = {'weighted_f1': float(f1_score(y_test, pred, average='weighted')), 'per_class_f1': f1_score(y_test, pred, average=None, labels=range(6))}
    RES[seed] = {'clean_f1': float(f1_all), 'additivity_err': err, 'imp': imp, 'signed': signed, 'glob_imp': glob_imp, 'subsets': sub}
    joblib.dump({'fp': cfg_fp, 'res': RES[seed]}, pf)
    print(f'  clean F1 {f1_all:.4f} | SHAP additivity max error {err:.2e} | subsets: ' + ', '.join(f'{k} {v["weighted_f1"]:.3f}' for k, v in sub.items()) + f' | {(time.time() - t0) / 60:.1f} min', flush=True)
print(f'Total {(time.time() - t_all) / 60:.1f} min')

## 4. Where the attribution sits: feature groups per class (mean +/- std over seeds)

In [ ]:
def share_by(groups_fn, imp):          # imp (6, F) -> (6, n_groups) fractions of each class's total attribution
    labels = groups_fn; out = np.stack([[imp[c, labels == g].sum() for g in np.unique(labels)] for c in range(6)])
    return out / out.sum(1, keepdims=True)
PC = np.where(IS_CYBER, 'cyber', 'physical')
share_fine = np.stack([share_by_ for share_by_ in [np.stack([[RES[s]['imp'][c, FEAT_GROUP == g].sum() for g in GROUP_ORDER] for c in range(6)]) for s in SEEDS]])
share_fine = share_fine / share_fine.sum(2, keepdims=True)                                       # (seeds, 6, groups)
share_pc = np.stack([[[RES[s]['imp'][c, ~IS_CYBER].sum(), RES[s]['imp'][c, IS_CYBER].sum()] for c in range(6)] for s in SEEDS]); share_pc = share_pc / share_pc.sum(2, keepdims=True)
rows = []
for c, cn in enumerate(class_names):
    r = {'class': cn, 'physical share': f'{share_pc[:, c, 0].mean():.3f} +/- {share_pc[:, c, 0].std(ddof=1):.3f}', 'cyber share': f'{share_pc[:, c, 1].mean():.3f} +/- {share_pc[:, c, 1].std(ddof=1):.3f}'}
    for gi, g in enumerate(GROUP_ORDER): r[g] = f'{share_fine[:, c, gi].mean():.3f}'
    rows.append(r)
share_table = pd.DataFrame(rows); pd.set_option('display.width', 260); print(share_table.to_string(index=False))
print('\nFeatures per group:', {g: int((FEAT_GROUP == g).sum()) for g in GROUP_ORDER})
print('Reference: if all 93 features were used equally, the physical share would be', f'{(~IS_CYBER).mean():.3f}', 'and the cyber share', f'{IS_CYBER.mean():.3f}')

## 5. Top-10 features per class (mean over seeds) and their shift from benign

In [ ]:
imp_mean = np.mean([RES[s]['imp'] for s in SEEDS], 0); imp_sd = np.std([RES[s]['imp'] for s in SEEDS], 0, ddof=1)
imp_share = imp_mean / imp_mean.sum(1, keepdims=True)
top_rows = []
for c, cn in enumerate(class_names):
    for rank, f in enumerate(np.argsort(-imp_mean[c])[:10], 1):
        top_rows.append({'class': cn, 'rank': rank, 'feature': FEATURE_COLS[f], 'group': FEAT_GROUP[f], 'share of class attribution': round(float(imp_share[c, f]), 4),
                         'mean |SHAP|': round(float(imp_mean[c, f]), 4), 'std over seeds': round(float(imp_sd[c, f]), 4),
                         'shift from benign (sigma)': round(float(SHIFT[c, f]), 3) if c > 0 else np.nan})
top_table = pd.DataFrame(top_rows)
for cn in class_names: print(f'\n--- {cn}'); print(top_table[top_table['class'] == cn].drop(columns='class').to_string(index=False))

## 6. Stability across seeds

In [ ]:
stab = []
for c, cn in enumerate(class_names):
    rho, ov = [], []
    for i, a in enumerate(SEEDS):
        for b in SEEDS[i + 1:]:
            va, vb = RES[a]['imp'][c], RES[b]['imp'][c]; keep = ~const_mask
            rho.append(stats.spearmanr(va[keep], vb[keep]).correlation); ov.append(len(set(np.argsort(-va)[:10]) & set(np.argsort(-vb)[:10])) / 10)
    stab.append({'class': cn, 'mean pairwise Spearman (non-constant features)': round(float(np.mean(rho)), 3), 'min': round(float(np.min(rho)), 3), 'top-10 overlap (mean)': round(float(np.mean(ov)), 2)})
stab_table = pd.DataFrame(stab); print(stab_table.to_string(index=False))
print('\nSHAP additivity max error over seeds:', max(RES[s]['additivity_err'] for s in SEEDS))

## 7. Necessity check: tuned models on physical-only / cyber-only / all features (per-class F1, test set)

In [ ]:
sub_rows = []
for name in SUBSETS:
    pcf = np.stack([RES[s]['subsets'][name]['per_class_f1'] for s in SEEDS]); wf = np.array([RES[s]['subsets'][name]['weighted_f1'] for s in SEEDS])
    r = {'feature set': name, 'weighted F1': f'{wf.mean():.4f} +/- {wf.std(ddof=1):.4f}'}
    r.update({cn: f'{pcf[:, i].mean():.3f} +/- {pcf[:, i].std(ddof=1):.3f}' for i, cn in enumerate(class_names)}); sub_rows.append(r)
subset_table = pd.DataFrame(sub_rows); print(subset_table.to_string(index=False))
print('\nF1 lost when the 60 cyber features are removed (all - physical only), per class:')
loss = np.stack([RES[s]['subsets']['all features']['per_class_f1'] - RES[s]['subsets']['physical only (33)']['per_class_f1'] for s in SEEDS])
print({cn: f'{loss[:, i].mean():+.3f}' for i, cn in enumerate(class_names)})
print('F1 lost when the 33 physical features are removed (all - cyber only), per class:')
loss2 = np.stack([RES[s]['subsets']['all features']['per_class_f1'] - RES[s]['subsets']['cyber only (60)']['per_class_f1'] for s in SEEDS])
print({cn: f'{loss2[:, i].mean():+.3f}' for i, cn in enumerate(class_names)})

## 8. Figure and save

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.8)); bottom = np.zeros(6); cols = plt.cm.tab10(np.arange(6))
for gi, g in enumerate(GROUP_ORDER):
    v = share_fine[:, :, gi].mean(0); ax[0].bar(class_names, v, bottom=bottom, label=g, color=cols[gi]); bottom += v
ax[0].set_ylabel('share of class attribution (mean |SHAP|)'); ax[0].set_title('Where the attribution sits, per true class'); ax[0].legend(fontsize=8, loc='upper center', ncol=3, bbox_to_anchor=(0.5, -0.12)); ax[0].tick_params(axis='x', rotation=20)
w = 0.27
for k, name in enumerate(SUBSETS):
    pcf = np.stack([RES[s]['subsets'][name]['per_class_f1'] for s in SEEDS]); ax[1].bar(np.arange(6) + (k - 1) * w, pcf.mean(0), w, yerr=pcf.std(0, ddof=1), capsize=2, label=name)
ax[1].set_xticks(np.arange(6)); ax[1].set_xticklabels(class_names, rotation=20); ax[1].set_ylabel('test F1'); ax[1].set_title('Per-class F1 by feature set (tuned XGBoost)'); ax[1].legend(fontsize=8); ax[1].grid(alpha=.3, axis='y')
plt.tight_layout(); plt.savefig(f'shap_per_class_{PIPELINE_VERSION}.png', dpi=150, bbox_inches='tight'); plt.show()

joblib.dump({'PIPELINE_VERSION': PIPELINE_VERSION, 'SEEDS': SEEDS, 'N_PER_CLASS': N_PER_CLASS, 'FEATURE_COLS': FEATURE_COLS, 'FEAT_GROUP': FEAT_GROUP, 'class_names': class_names,
             'share_table': share_table, 'top_table': top_table, 'stability_table': stab_table, 'subset_table': subset_table, 'imp_mean': imp_mean, 'imp_sd': imp_sd, 'shift_from_benign': SHIFT,
             'per_seed': {s: {k: v for k, v in RES[s].items()} for s in SEEDS}}, f'shap_results_{PIPELINE_VERSION}.pkl')
share_table.to_csv(f'shap_group_share_{PIPELINE_VERSION}.csv', index=False); top_table.to_csv(f'shap_top10_per_class_{PIPELINE_VERSION}.csv', index=False)
stab_table.to_csv(f'shap_stability_{PIPELINE_VERSION}.csv', index=False); subset_table.to_csv(f'shap_feature_subset_f1_{PIPELINE_VERSION}.csv', index=False)
pd.DataFrame(imp_mean.T, index=FEATURE_COLS, columns=class_names).to_csv(f'shap_mean_abs_all_features_{PIPELINE_VERSION}.csv')
json.dump({'notebook': 'NB-SHAP', 'pipeline_version': PIPELINE_VERSION, 'seeds': SEEDS, 'n_per_class': N_PER_CLASS, 'xgboost': xgb.__version__, 'backend': BACKEND_NAME,
           'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}, open(f'nb_shap_run_manifest_{PIPELINE_VERSION}.json', 'w'), indent=2)
for f in [f'shap_results_{PIPELINE_VERSION}.pkl', f'shap_group_share_{PIPELINE_VERSION}.csv', f'shap_top10_per_class_{PIPELINE_VERSION}.csv', f'shap_stability_{PIPELINE_VERSION}.csv',
          f'shap_feature_subset_f1_{PIPELINE_VERSION}.csv', f'shap_mean_abs_all_features_{PIPELINE_VERSION}.csv', f'shap_per_class_{PIPELINE_VERSION}.png', f'nb_shap_run_manifest_{PIPELINE_VERSION}.json']:
    print(('  OK      ' if os.path.exists(f) else '  MISSING ') + f)